# 🐼 The Easy Pandas Guide: From Zero to Data Masterclass
> **A Comprehensive, Friendly, and Deeply Detailed Reference Guide to Tabular Data**
> 
> *Target Audience:* Beginners looking for crystal-clear clarity, Data Analysts transitioning from Excel/SQL, and ML Engineers engineering clean tabular features.

---

### 🌟 How to Use This Notebook
1. **Detailed Text & Explanations:** Every single concept is explained thoroughly in plain English, with detailed background context, intuitive analogies, and "under-the-hood" mechanics.
2. **Visual Mental Models:** Comprehensive ASCII diagrams illustrate how Series, DataFrames, Indexing, and the GroupBy Split-Apply-Combine engine work.
3. **Pre-Executed & Verified:** Every code cell is pre-executed with real outputs so you can read this like an illustrated technical book or run each cell interactively to experiment.
4. **Pro-Tips, Gotchas & Best Practices:** Dedicated callout blocks highlight real-world gotchas like `SettingWithCopyWarning`, NaN comparisons, and memory optimization.
5. **Smooth Navigation:** Use the **Table of Contents** below or each chapter's `[⬆️ Back to Table of Contents]` link to jump directly to any section.

---

<a id="table-of-contents"></a>
## 📑 Table of Contents

| Part | Chapter | Key Topics Covered |
| :---: | :--- | :--- |
| **I** | [1. Core Mental Model: What is Pandas?](#ch1) | Python Lists vs NumPy vs Pandas, DataFrame Anatomy, Hash-table Indexing |
| **I** | [2. Creating Series & DataFrames](#ch2) | From Lists, Dictionaries, NumPy Arrays; Custom Indices, Automatic Alignment |
| **I** | [3. Reading, Writing & Inspecting Data](#ch3) | CSV, Excel, Parquet, JSON; `head()`, `info()`, `describe()`, memory usage |
| **II** | [4. Selection & Slicing: loc vs iloc](#ch4) | `[]` syntax, `iloc` (position) vs `loc` (labels), boundary rules, scalar `.at`/`.iat` |
| **II** | [5. Filtering & Boolean Queries](#ch5) | Conditions (`&`, `\|`, `~`), `.isin()`, `.between()`, `.query()`, avoiding `and`/`or` |
| **III** | [6. Modifying: Adding, Renaming & Dropping](#ch6) | Vectorized column creation, `.assign()`, `.rename()`, `.drop()`, axis model |
| **III** | [7. Handling Missing Data (NaNs)](#ch7) | `np.nan` mechanics, `.isna()`, `.notna()`, `.dropna()`, `.fillna()`, imputation |
| **III** | [8. Sorting, Ranking & Index Manipulation](#ch8) | `.sort_values()`, multi-column sort, `.rank()`, `.reset_index()`, `.set_index()` |
| **IV** | [9. Summary Statistics & Frequencies](#ch9) | Aggregations, `.value_counts()`, `.nunique()`, correlation `.corr()` |
| **IV** | [10. The GroupBy Engine (Split-Apply-Combine)](#ch10) | Mental model, `.agg()`, multi-level aggregations, `.transform()`, `.filter()` |
| **V** | [11. Text & String Operations (`.str`)](#ch11) | Cleaning whitespace, case changes, regex pattern matching, `.split(expand=True)` |
| **V** | [12. Working with Dates & Times (`.dt`)](#ch12) | `pd.to_datetime()`, calendar properties, elapsed time, filtering date ranges |
| **V** | [13. Combining Datasets: Concat & Merge](#ch13) | Stacking (`concat`), SQL joins (`merge`: inner, left, right, outer, anti-join) |
| **V** | [14. Reshaping: Pivot Tables & Melt](#ch14) | Cross-tabulations with `pivot_table()`, unpivoting wide data with `melt()` |
| **V** | [15. Performance, Gotchas & Cheat Sheet](#ch15) | `SettingWithCopyWarning`, vectorization benchmark vs loops, categorical dtypes |

<a id="ch1"></a>
---
# Part I: Foundations & Mental Models

## 1. Core Mental Model: What is Pandas?

In scientific and business computing, data rarely arrives as a clean, uniform matrix of floating-point numbers. Instead, real-world data is **heterogeneous** and **tabular**:
- A single record might contain a person's **name** (text), **age** (integer), **salary** (floating-point), **hire date** (datetime), and **department** (category).
- Rows represent individual entities (e.g., customers, transactions, sensor readings).
- Columns represent variables or attributes measured for each entity.

### 1.1 Python Lists vs. NumPy ndarrays vs. Pandas DataFrames
To appreciate why Pandas is the universal standard for tabular data, let's contrast how Python tools handle data:

| Feature | Standard Python List of Dicts | NumPy ndarray | Pandas DataFrame |
| :--- | :--- | :--- | :--- |
| **Data Types** | Mixed (dynamic types) | Homogeneous (single dtype across entire array) | Heterogeneous (each column has its own dtype) |
| **Row / Column Labels** | No native labels; manual dictionary key lookups | Integer positions only `[0, 1, 2]` | Dual-axis labels: named columns AND indexed rows |
| **Speed & Vectorization** | Slow; requires Python `for` loops | Blazing fast; contiguous C/SIMD arrays | Blazing fast; built directly on top of NumPy |
| **Missing Data Support** | Manual `None` checks | `np.nan` (floats only; clumsy for strings/ints) | First-class null handling across all types |
| **Relational Operations** | Complex custom nested loops | Not supported natively | SQL-like Joins, Merges, GroupBys, Pivot Tables |

### 1.2 The Anatomy of a DataFrame
A Pandas **DataFrame** is a 2-dimensional labeled data structure composed of three interconnected parts:
1. **The Index (Row Labels):** Metadata identifying each row (e.g., 0, 1, 2 or customer IDs like `'CUST-001'`).
2. **The Columns (Feature Labels):** Metadata identifying each variable (e.g., `'Name'`, `'Salary'`).
3. **The Series (Column Data):** Each column is a 1-dimensional **`pd.Series`** containing the actual data buffer.

```
                                  COLUMNS (Variables)
                    'Name'          'Age'         'Department'        'Salary'
               +---------------+-------------+--------------------+--------------+
 Index 0 ----> |    Alice      |     25      |         HR         |   60,000.0   |  \
 Index 1 ----> |    Bob        |     30      |         IT         |   85,000.0   |   |-- ROWS
 Index 2 ----> |    Charlie    |     35      |      Marketing     |   72,000.0   |  /   (Records)
               +---------------+-------------+--------------------+--------------+
                   pd.Series       pd.Series       pd.Series          pd.Series
                  dtype: object   dtype: int64    dtype: object      dtype: float64
```

### 1.3 Under the Hood: The Index Hash Map
Why does Pandas have an **Index** instead of just using plain row numbers?
- Under the hood, the Pandas Index is implemented using an efficient hash table.
- When you look up a row by its label (e.g., `df.loc['CUST-104']`), Pandas does not scan rows one by one in $O(N)$ time. Instead, it performs an instant $O(1)$ hash map lookup to locate the exact memory offset of that record!

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

Pandas Version: 3.0.6
NumPy Version:  2.5.3


<a id="ch2"></a>
---
## 2. Creating Series & DataFrames from Scratch

Before loading massive external CSV or database files, mastering how to construct Series and DataFrames from native Python objects is essential for creating test cases, mocking data, and converting computation outputs.

### 2.1 The `pd.Series`: 1D Labeled Array
A `Series` is the foundational building block of Pandas. Think of it as a 1D NumPy array with a labeled index.
- If you don't supply an index, Pandas automatically assigns a default `RangeIndex` starting at `0, 1, 2, ...`.
- If you supply custom labels, you can access elements using both the label name and its zero-based integer position.

In [2]:
# 1. Series from a Python list with default integer index
temperatures = pd.Series([22.5, 24.0, 19.8, 25.2, 21.0], name='Temperature_C')
print("Series with default RangeIndex:")
print(temperatures)

# 2. Series with custom index labels (e.g. Days of the week)
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
weekly_temps = pd.Series([22.5, 24.0, 19.8, 25.2, 21.0], index=days, name='Daily_Temp')
print("\nSeries with custom day labels:")
print(weekly_temps)

# Lookups: By label and by position
print(f"\nWednesday's temp (by label):    {weekly_temps['Wed']}°C")
print(f"First day's temp (by position): {weekly_temps.iloc[0]}°C")

Series with default RangeIndex:
0    22.5
1    24.0
2    19.8
3    25.2
4    21.0
Name: Temperature_C, dtype: float64

Series with custom day labels:
Mon    22.5
Tue    24.0
Wed    19.8
Thu    25.2
Fri    21.0
Name: Daily_Temp, dtype: float64

Wednesday's temp (by label):    19.8°C
First day's temp (by position): 22.5°C


### 2.2 Automatic Index Alignment: A Series Superpower
One of the most powerful and unique features of Pandas is **automatic index alignment**.
When performing arithmetic on two Series (like addition, subtraction, or multiplication), Pandas does **not** simply pair elements by their physical order. Instead, it automatically matches elements by their **index labels**!
If an index label exists in one Series but not the other, Pandas gracefully inserts `NaN` (Not a Number) to prevent silent misalignments.

In [3]:
# Sales in Week 1 and Week 2 with different product assortments
week1 = pd.Series({'Apples': 50, 'Bananas': 30, 'Cherries': 20})
week2 = pd.Series({'Bananas': 45, 'Apples': 60, 'Dates': 15})

print("Week 1 Sales:\n", week1)
print("\nWeek 2 Sales:\n", week2)

# Vectorized addition aligns by product name automatically!
total_sales = week1 + week2
print("\nTotal Combined Sales (Notice automatic alignment & NaN for unmatched):")
print(total_sales)

Week 1 Sales:
 Apples      50
Bananas     30
Cherries    20
dtype: int64

Week 2 Sales:
 Bananas    45
Apples     60
Dates      15
dtype: int64

Total Combined Sales (Notice automatic alignment & NaN for unmatched):
Apples      110.0
Bananas      75.0
Cherries      NaN
Dates         NaN
dtype: float64


### 2.3 Creating a `pd.DataFrame`
There are several intuitive ways to create a DataFrame:

#### Method A: Dictionary of Lists (Column-Oriented)
This is the most common format: keys become column headers, and list values become column elements. All lists must have identical lengths.

In [4]:
# Method A: Dictionary of Lists
staff_data = {
    'Emp_ID': [101, 102, 103, 104, 105],
    'Name': ['Alice Smith', 'Bob Jones', 'Charlie Brown', 'Diana Prince', 'Evan Wright'],
    'Department': ['HR', 'IT', 'Marketing', 'IT', 'Finance'],
    'Salary': [62000, 88000, 71000, 94000, 105000],
    'Years_Experience': [3, 7, 4, 9, 12],
    'Is_Remote': [True, True, False, False, True]
}

df_staff = pd.DataFrame(staff_data)
print("DataFrame created from Dictionary of Lists:")
display(df_staff)

DataFrame created from Dictionary of Lists:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
0,101,Alice Smith,HR,62000,3,True
1,102,Bob Jones,IT,88000,7,True
2,103,Charlie Brown,Marketing,71000,4,False
3,104,Diana Prince,IT,94000,9,False
4,105,Evan Wright,Finance,105000,12,True


#### Method B: List of Dictionaries (Row-Oriented)
This format mirrors records returned by REST APIs, JSON files, or database queries where each item represents an individual record.

In [5]:
# Method B: List of Dictionaries (Row-oriented)
api_records = [
    {'City': 'New York', 'Country': 'USA', 'Population_M': 8.3},
    {'City': 'Tokyo', 'Country': 'Japan', 'Population_M': 13.9},
    {'City': 'Paris', 'Country': 'France', 'Population_M': 2.1}
]

df_cities = pd.DataFrame(api_records)
print("DataFrame created from List of Dictionaries (API/JSON format):")
display(df_cities)

DataFrame created from List of Dictionaries (API/JSON format):


,City,Country,Population_M
0,New York,USA,8.3
1,Tokyo,Japan,13.9
2,Paris,France,2.1


#### Method C: From a 2D NumPy Array
When converting matrix computations into a labeled table, provide explicit `columns` and `index` arguments:

In [6]:
# Method C: From 2D NumPy array
np.random.seed(42)
matrix = np.random.randint(10, 100, size=(4, 3))

df_matrix = pd.DataFrame(
    matrix,
    columns=['Feature_A', 'Feature_B', 'Feature_C'],
    index=['Sample_1', 'Sample_2', 'Sample_3', 'Sample_4']
)
print("DataFrame created from 2D NumPy Array:")
display(df_matrix)

DataFrame created from 2D NumPy Array:


,Feature_A,Feature_B,Feature_C
Sample_1,61,24,81
Sample_2,70,30,92
Sample_3,96,84,84
Sample_4,97,33,12


### 2.4 DataFrame Anatomy & Structural Attributes
Once a DataFrame is created, you can inspect its structural attributes without parentheses:
- **`.shape`**: A tuple `(n_rows, n_columns)`.
- **`.columns`**: An `Index` object containing column names.
- **`.index`**: An `Index` object containing row labels.
- **`.dtypes`**: The data type assigned to each column.
- **`.ndim`**: The dimensionality (always `2` for a DataFrame, `1` for a Series).
- **`.size`**: Total number of cells (`rows * columns`).

In [7]:
print(f"Shape (rows, cols): {df_staff.shape}")
print(f"Total cells:        {df_staff.size}")
print(f"Dimensions:         {df_staff.ndim}")
print(f"Columns:            {list(df_staff.columns)}")
print(f"Row Index:          {df_staff.index}")
print("\nColumn Data Types:")
print(df_staff.dtypes)

Shape (rows, cols): (5, 6)
Total cells:        30
Dimensions:         2
Columns:            ['Emp_ID', 'Name', 'Department', 'Salary', 'Years_Experience', 'Is_Remote']
Row Index:          RangeIndex(start=0, stop=5, step=1)

Column Data Types:
Emp_ID              int64
Name                  str
Department            str
Salary              int64
Years_Experience    int64
Is_Remote            bool
dtype: object


> 💡 **Key Takeaway:**
> Every DataFrame is a collection of Series. Each column is homogeneous (one dtype), but the DataFrame as a whole is heterogeneous (contains diverse dtypes).
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch3"></a>
---
## 3. Reading, Writing & Inspecting Data

In professional workflows, over 95% of datasets are imported from external storage: CSV files, Excel spreadsheets, SQL databases, Parquet files, or Cloud Object Stores.

### 3.1 Common I/O Functions in Pandas
Pandas provides bidirectional readers and writers for virtually every data format:

| Format | Read Function | Write Function | Key Advantages & Notes |
| :--- | :--- | :--- | :--- |
| **CSV** | `pd.read_csv()` | `df.to_csv()` | Universal, human-readable text format. |
| **Excel** | `pd.read_excel()` | `df.to_excel()` | Supports multi-sheet `.xlsx` workbooks. |
| **Parquet** | `pd.read_parquet()` | `df.to_parquet()` | Fast, columnar, compressed binary; ideal for Big Data & ML. |
| **JSON** | `pd.read_json()` | `df.to_json()` | Standard web format; handles nested dictionaries. |
| **SQL** | `pd.read_sql()` | `df.to_sql()` | Connects directly to PostgreSQL, MySQL, SQLite, etc. |

### 3.2 Key Arguments in `pd.read_csv()`
Understanding these arguments avoids common parsing headaches:
- `filepath_or_buffer`: Path to local file or URL.
- `sep` or `delimiter`: Character separating values (default is `','`, use `'\t'` for tab-separated).
- `index_col`: Column to use as the row index (e.g., `index_col='User_ID'`).
- `usecols`: List of specific columns to load (saves memory when files have 100+ columns).
- `parse_dates`: List of column names to parse directly as datetimes instead of plain strings.
- `na_values`: Additional strings to interpret as `NaN` (e.g. `['N/A', 'missing', '-999']`).
- `chunksize`: Number of rows to read per iteration when processing files larger than system RAM.

In [8]:
# Export df_staff to a CSV file to demonstrate file I/O
csv_filename = "staff_sample.csv"
df_staff.to_csv(csv_filename, index=False)
print(f"Exported staff data to {csv_filename} successfully!")

# Read it back with specific parameters
loaded_df = pd.read_csv(csv_filename)
print("Reloaded DataFrame from CSV:")
display(loaded_df.head(2))

# Clean up temporary file
import os
if os.path.exists(csv_filename):
    os.remove(csv_filename)

Exported staff data to staff_sample.csv successfully!
Reloaded DataFrame from CSV:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
0,101,Alice Smith,HR,62000,3,True
1,102,Bob Jones,IT,88000,7,True


### 3.3 Exploratory Data Inspection: First Look at Any Dataset
Whenever you receive a dataset, always execute these inspection methods first:

1. **`df.head(n)`**: Displays the first $n$ rows (default is 5). Verifies column headers and initial values.
2. **`df.tail(n)`**: Displays the last $n$ rows. Useful to check for corrupted final rows or trailing summaries.
3. **`df.sample(n)`**: Displays $n$ random rows. Prevents bias if data is sorted.
4. **`df.info()`**: Comprehensive technical summary: total rows, column names, non-null counts, dtypes, and memory usage.
5. **`df.describe()`**: Statistical summary of numeric distributions (mean, std, min, quartiles, max).

In [9]:
# 1. Preview head, tail, and random sample
print("--- Head (First 3 rows) ---")
display(df_staff.head(3))

print("\n--- Tail (Last 2 rows) ---")
display(df_staff.tail(2))

print("\n--- Random Sample (2 rows) ---")
display(df_staff.sample(2, random_state=42))

--- Head (First 3 rows) ---


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
0,101,Alice Smith,HR,62000,3,True
1,102,Bob Jones,IT,88000,7,True
2,103,Charlie Brown,Marketing,71000,4,False



--- Tail (Last 2 rows) ---


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
3,104,Diana Prince,IT,94000,9,False
4,105,Evan Wright,Finance,105000,12,True



--- Random Sample (2 rows) ---


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
1,102,Bob Jones,IT,88000,7,True
4,105,Evan Wright,Finance,105000,12,True


In [10]:
# 2. Technical diagnostic with .info()
print("--- DataFrame Technical Diagnostics ---")
df_staff.info()

--- DataFrame Technical Diagnostics ---
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Emp_ID            5 non-null      int64
 1   Name              5 non-null      str  
 2   Department        5 non-null      str  
 3   Salary            5 non-null      int64
 4   Years_Experience  5 non-null      int64
 5   Is_Remote         5 non-null      bool 
dtypes: bool(1), int64(3), str(2)
memory usage: 337.0 bytes


In [11]:
# 3. Statistical summary with .describe()
print("--- Numerical Columns Statistical Summary ---")
display(df_staff.describe())

# Include non-numeric / categorical columns too
print("\n--- All Columns Summary (including text/categorical) ---")
display(df_staff.describe(include='all'))

--- Numerical Columns Statistical Summary ---


,Emp_ID,Salary,Years_Experience
count,5.000000,5.000000,5.000000
mean,103.000000,84000.000000,7.000000
std,1.581139,17392.527131,3.674235
min,101.000000,62000.000000,3.000000
25%,102.000000,71000.000000,4.000000
50%,103.000000,88000.000000,7.000000
75%,104.000000,94000.000000,9.000000
max,105.000000,105000.000000,12.000000



--- All Columns Summary (including text/categorical) ---


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
count,5.000000,5,5,5.000000,5.000000,5
unique,NaN,5,4,NaN,NaN,2
top,NaN,Alice Smith,IT,NaN,NaN,True
freq,NaN,1,2,NaN,NaN,3
mean,103.000000,NaN,NaN,84000.000000,7.000000,NaN
std,1.581139,NaN,NaN,17392.527131,3.674235,NaN
min,101.000000,NaN,NaN,62000.000000,3.000000,NaN
25%,102.000000,NaN,NaN,71000.000000,4.000000,NaN
50%,103.000000,NaN,NaN,88000.000000,7.000000,NaN
75%,104.000000,NaN,NaN,94000.000000,9.000000,NaN


> 💡 **Pro-Tip:**
> When working with large datasets, run `df.info(memory_usage='deep')` to measure the **true** memory consumption including dynamic Python string allocations!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch4"></a>
---
# Part II: Slicing, Indexing & Filtering

## 4. Selection & Slicing: `loc` vs `iloc`

Selecting specific rows and columns is the most frequent operation in data analysis. Beginners often struggle with when to use `[]`, `.loc[]`, or `.iloc[]`.

Here is the unambiguous, foolproof rule:

```
+-----------------------------------------------------------------------------------------------+
|                             SELECTION CHEAT SHEET & MENTAL MODEL                              |
+-----------------------------------------------------------------------------------------------+
|  Method  | Based On...          | What to Pass                   | Endpoint Inclusion?        |
+----------+----------------------+--------------------------------+----------------------------+
|  df['C'] | Column Label         | Column name or list of names   | N/A (Columns only)         |
|  .iloc   | Integer Position     | 0-based indices [row, col]     | EXCLUSIVE stop (0:3 -> 0,1,2)|
|  .loc    | Label Names          | Row & column names [row, col]  | INCLUSIVE stop ('A':'C')   |
|  .iat    | Integer Position     | Scalar position [row, col]     | N/A (Fast single cell)     |
|  .at     | Label Name           | Scalar label [row, col]        | N/A (Fast single cell)     |
+-----------------------------------------------------------------------------------------------+
```

### 4.1 Column Selection: Series vs DataFrame
- **Single Bracket `df['Name']`**: Returns a 1D **`Series`**.
- **Double Bracket `df[['Name', 'Salary']]`**: Returns a 2D **`DataFrame`** (even if you only request one column like `df[['Name']]`).

In [12]:
# Single bracket returns a Series
name_series = df_staff['Name']
print(f"Type of df_staff['Name']: {type(name_series)}")
print(name_series.head(2))

# Double bracket returns a DataFrame
sub_df = df_staff[['Name', 'Department', 'Salary']]
print(f"\nType of df_staff[['Name', 'Salary']]: {type(sub_df)}")
display(sub_df.head(3))

Type of df_staff['Name']: <class 'pandas.Series'>
0    Alice Smith
1      Bob Jones
Name: Name, dtype: str

Type of df_staff[['Name', 'Salary']]: <class 'pandas.DataFrame'>


,Name,Department,Salary
0,Alice Smith,HR,62000
1,Bob Jones,IT,88000
2,Charlie Brown,Marketing,71000


### 4.2 Positional Selection with `.iloc` (Integer-Location)
`.iloc` works strictly with zero-based integer indices: `.iloc[row_selection, column_selection]`.
It strictly follows standard Python slicing semantics: **the stop endpoint is excluded**!

In [13]:
# 1. Single row by integer position (returns Series)
print("Row at index position 0 (First Employee):")
print(df_staff.iloc[0])

# 2. Slice rows 0 to 3 (exclusive -> rows 0, 1, 2) and columns 1 to 4
print("\nRows 0:3 and Columns 1:4:")
display(df_staff.iloc[0:3, 1:4])

# 3. Discontinuous selection using integer lists
print("\nRows [0, 2, 4] and Columns [1, 3]:")
display(df_staff.iloc[[0, 2, 4], [1, 3]])

# 4. Negative integer indexing (Last 2 rows, last 3 columns)
print("\nLast 2 rows, last 3 columns:")
display(df_staff.iloc[-2:, -3:])

Row at index position 0 (First Employee):
Emp_ID                      101
Name                Alice Smith
Department                   HR
Salary                    62000
Years_Experience              3
Is_Remote                  True
Name: 0, dtype: object

Rows 0:3 and Columns 1:4:


,Name,Department,Salary
0,Alice Smith,HR,62000
1,Bob Jones,IT,88000
2,Charlie Brown,Marketing,71000



Rows [0, 2, 4] and Columns [1, 3]:


,Name,Salary
0,Alice Smith,62000
2,Charlie Brown,71000
4,Evan Wright,105000



Last 2 rows, last 3 columns:


,Salary,Years_Experience,Is_Remote
3,94000,9,False
4,105000,12,True


### 4.3 Label-Based Selection with `.loc` (Label-Location)
`.loc` selects data by the **actual names** of the rows and columns: `.loc[row_label, column_label]`.
**Crucial Rule:** Unlike standard Python slicing, when slicing with `.loc`, **the stop label is INCLUDED**!

In [14]:
# Set employee name as the DataFrame index to showcase .loc
df_named = df_staff.set_index('Name')
print("DataFrame indexed by Employee Name:")
display(df_named)

# 1. Select a single row by label
print("\nCharlie Brown's Record:")
print(df_named.loc['Charlie Brown'])

# 2. Select specific row and specific columns
print(f"\nDiana Prince's Salary: ${df_named.loc['Diana Prince', 'Salary']:,}")

# 3. Slice from 'Bob Jones' to 'Diana Prince' (Diana IS included!)
print("\nSlice from 'Bob Jones' to 'Diana Prince':")
display(df_named.loc['Bob Jones':'Diana Prince', ['Department', 'Salary', 'Years_Experience']])

DataFrame indexed by Employee Name:


,Emp_ID,Department,Salary,Years_Experience,Is_Remote
Name,,,,,
Alice Smith,101,HR,62000,3,True
Bob Jones,102,IT,88000,7,True
Charlie Brown,103,Marketing,71000,4,False
Diana Prince,104,IT,94000,9,False
Evan Wright,105,Finance,105000,12,True



Charlie Brown's Record:
Emp_ID                    103
Department          Marketing
Salary                  71000
Years_Experience            4
Is_Remote               False
Name: Charlie Brown, dtype: object

Diana Prince's Salary: $94,000

Slice from 'Bob Jones' to 'Diana Prince':


,Department,Salary,Years_Experience
Name,,,
Bob Jones,IT,88000,7
Charlie Brown,Marketing,71000,4
Diana Prince,IT,94000,9


### 4.4 Ultra-Fast Scalar Lookups with `.at` and `.iat`
When you need to read or update a single cell inside a high-throughput loop, `.loc` and `.iloc` incur extra overhead for series validation.
- **`.at[row_label, col_label]`**: Fast scalar access by label.
- **`.iat[row_idx, col_idx]`**: Fast scalar access by integer position.

In [15]:
# Fast scalar lookup
val_at = df_named.at['Alice Smith', 'Salary']
val_iat = df_staff.iat[0, 3] # Row 0, Col 3 ('Salary')

print(f"Scalar lookup via .at:  ${val_at:,}")
print(f"Scalar lookup via .iat: ${val_iat:,}")

Scalar lookup via .at:  $62,000
Scalar lookup via .iat: $62,000


> ⚠️ **Common Trap:**
> Slicing with `.iloc[0:2]` returns 2 rows (indices 0 and 1).
> Slicing with `.loc[0:2]` on a default integer index returns 3 rows (indices 0, 1, and 2)!
> Always remember: `.loc` includes both boundaries!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch5"></a>
---
## 5. Filtering & Boolean Queries

Filtering extracts records that satisfy business criteria (e.g. "Find all remote employees in IT earning over $80,000").

### 5.1 How Boolean Indexing Works Under the Hood
1. When you write `df_staff['Salary'] > 75000`, Pandas applies the comparison operator element-wise to every row.
2. This generates a **Boolean Mask** (a Series of `True` and `False` values).
3. When you pass this mask back into `df_staff[mask]`, Pandas retains only rows where the mask evaluates to `True`.

In [16]:
# 1. Generate the boolean mask
mask_high_salary = df_staff['Salary'] > 75000
print("Boolean Mask:")
print(mask_high_salary)

# 2. Apply mask to filter DataFrame
print("\nEmployees earning > $75k:")
display(df_staff[mask_high_salary])

Boolean Mask:
0    False
1     True
2    False
3     True
4     True
Name: Salary, dtype: bool

Employees earning > $75k:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
1,102,Bob Jones,IT,88000,7,True
3,104,Diana Prince,IT,94000,9,False
4,105,Evan Wright,Finance,105000,12,True


### 5.2 Combining Multiple Conditions: The 3 Rules
When combining conditions in Pandas, standard Python keywords `and`, `or`, and `not` will **raise a `ValueError`**!
Instead, you must use bitwise operators:
- **`&`** for AND (both conditions must be true)
- **`|`** for OR (at least one condition must be true)
- **`~`** for NOT (negates the condition)

**The Golden Rule:** Every individual condition MUST be wrapped in parentheses `()`!
This is required because Python's bitwise operator precedence is higher than comparison operators (`>`, `==`, `<`). Without parentheses, `df['A'] > 5 & df['B'] < 10` is evaluated as `5 & df['B']`, causing a syntax crash.

In [17]:
# 1. AND Condition: Department == 'IT' AND Salary > 90,000
it_high_earners = df_staff[(df_staff['Department'] == 'IT') & (df_staff['Salary'] > 90000)]
print("IT Employees earning > $90k:")
display(it_high_earners)

# 2. OR Condition: Years_Experience >= 10 OR Is_Remote == True
senior_or_remote = df_staff[(df_staff['Years_Experience'] >= 10) | (df_staff['Is_Remote'] == True)]
print("\nSenior (>=10 yrs) OR Remote Employees:")
display(senior_or_remote)

# 3. NOT Condition: NOT in IT
non_it_staff = df_staff[~(df_staff['Department'] == 'IT')]
print("\nNon-IT Employees:")
display(non_it_staff)

IT Employees earning > $90k:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
3,104,Diana Prince,IT,94000,9,False



Senior (>=10 yrs) OR Remote Employees:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
0,101,Alice Smith,HR,62000,3,True
1,102,Bob Jones,IT,88000,7,True
4,105,Evan Wright,Finance,105000,12,True



Non-IT Employees:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
0,101,Alice Smith,HR,62000,3,True
2,103,Charlie Brown,Marketing,71000,4,False
4,105,Evan Wright,Finance,105000,12,True


### 5.3 Convenient Helper Filtering Methods

#### Method 1: `.isin()` for Multiple Category Matches
Instead of chaining endless `|` conditions (`(df['Dept'] == 'HR') | (df['Dept'] == 'IT') | ...`), pass a list to `.isin()`:

In [18]:
target_depts = ['HR', 'Finance']
selected_staff = df_staff[df_staff['Department'].isin(target_depts)]
print("Employees in HR or Finance:")
display(selected_staff)

Employees in HR or Finance:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
0,101,Alice Smith,HR,62000,3,True
4,105,Evan Wright,Finance,105000,12,True


#### Method 2: `.between(left, right)` for Range Checks
Replaces `(df['Salary'] >= 70000) & (df['Salary'] <= 90000)`. Both boundaries are inclusive by default:

In [19]:
mid_salary_staff = df_staff[df_staff['Salary'].between(70000, 90000)]
print("Employees with Salary between $70k and $90k:")
display(mid_salary_staff)

Employees with Salary between $70k and $90k:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
1,102,Bob Jones,IT,88000,7,True
2,103,Charlie Brown,Marketing,71000,4,False


### 5.4 SQL-Style Querying with `.query()`
Pandas provides `.query()`, allowing you to write filtering conditions as clean, legible strings.
To reference external Python variables inside `.query()`, prefix the variable name with `@`:

In [20]:
min_experience = 5
dept_target = 'IT'

# Using .query() with @variable reference
query_results = df_staff.query("Department == @dept_target and Years_Experience >= @min_experience")
print(f"Employees in {dept_target} with >= {min_experience} years experience (via .query):")
display(query_results)

Employees in IT with >= 5 years experience (via .query):


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote
1,102,Bob Jones,IT,88000,7,True
3,104,Diana Prince,IT,94000,9,False


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch6"></a>
---
# Part III: Data Modification & Cleaning

## 6. Modifying Data: Adding, Renaming & Dropping

### 6.1 Adding New Columns (Vectorized Arithmetic)
Adding calculated features in Pandas is simple: assign the result of a vectorized computation directly to `df['New_Column']`.
Because operations are vectorized, calculations execute in optimized C code across millions of rows simultaneously without any Python `for` loops.

In [21]:
# Work on a fresh copy
df_work = df_staff.copy()

# 1. Simple arithmetic: Annual bonus is 12% of salary
df_work['Bonus'] = df_work['Salary'] * 0.12

# 2. Multi-column calculation: Total compensation
df_work['Total_Compensation'] = df_work['Salary'] + df_work['Bonus']

# 3. Calculated boolean flag
df_work['Senior_Level'] = df_work['Years_Experience'] >= 8

# 4. Using .assign() for method chaining
df_work = df_work.assign(
    Monthly_Gross=lambda d: (d['Salary'] / 12).round(2)
)

print("DataFrame after adding calculated columns:")
display(df_work[['Name', 'Salary', 'Bonus', 'Total_Compensation', 'Monthly_Gross', 'Senior_Level']])

DataFrame after adding calculated columns:


,Name,Salary,Bonus,Total_Compensation,Monthly_Gross,Senior_Level
0,Alice Smith,62000,7440.0,69440.0,5166.67,False
1,Bob Jones,88000,10560.0,98560.0,7333.33,False
2,Charlie Brown,71000,8520.0,79520.0,5916.67,False
3,Diana Prince,94000,11280.0,105280.0,7833.33,True
4,Evan Wright,105000,12600.0,117600.0,8750.00,True


### 6.2 Renaming Columns & Indices
Use `.rename()` with a dictionary mapping `{'old_name': 'new_name'}`:

In [22]:
df_renamed = df_work.rename(columns={
    'Emp_ID': 'Employee_Identifier',
    'Is_Remote': 'Remote_Worker'
})
print("Renamed Columns:")
print(list(df_renamed.columns))

Renamed Columns:
['Employee_Identifier', 'Name', 'Department', 'Salary', 'Years_Experience', 'Remote_Worker', 'Bonus', 'Total_Compensation', 'Senior_Level', 'Monthly_Gross']


### 6.3 Dropping Columns and Rows
Use `.drop()`:
- Specify `columns=['Col1', 'Col2']` to drop columns.
- Specify `index=[0, 1]` to drop rows by index label.
- **The Axis Model:** In Pandas, `axis=0` refers to rows (vertical axis), and `axis=1` refers to columns (horizontal axis).

In [23]:
# Dropping columns
df_dropped_cols = df_work.drop(columns=['Bonus', 'Monthly_Gross'])
print("After dropping Bonus and Monthly_Gross:")
display(df_dropped_cols.head(2))

# Dropping rows by index label
df_dropped_rows = df_work.drop(index=[0, 2])
print("\nAfter dropping row indices 0 and 2:")
display(df_dropped_rows.head(2))

After dropping Bonus and Monthly_Gross:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote,Total_Compensation,Senior_Level
0,101,Alice Smith,HR,62000,3,True,69440.0,False
1,102,Bob Jones,IT,88000,7,True,98560.0,False



After dropping row indices 0 and 2:


,Emp_ID,Name,Department,Salary,Years_Experience,Is_Remote,Bonus,Total_Compensation,Senior_Level,Monthly_Gross
1,102,Bob Jones,IT,88000,7,True,10560.0,98560.0,False,7333.33
3,104,Diana Prince,IT,94000,9,False,11280.0,105280.0,True,7833.33


> 💡 **Best Practice Note on `inplace=True`:**
> You may encounter `df.drop(..., inplace=True)` in older tutorials. In modern Pandas, **`inplace=True` is officially discouraged and slated for future deprecation**.
> Why?
> 1. It rarely provides true memory savings (it usually makes a copy internally anyway).
> 2. It breaks method chaining (`df.drop(...).rename(...)`).
> 3. Direct assignment (`df = df.drop(...)`) makes data flow clear and bug-free!
>
> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch7"></a>
---
## 7. Handling Missing Data (NaNs)

Missing values are an unavoidable reality of real-world datasets: survey respondents skip questions, sensors disconnect, and database joins produce unmatched records.

In Pandas:
- Floating-point missing values are represented by **`np.nan`** (IEEE 754 floating-point Not-a-Number).
- Modern Pandas also supports **`pd.NA`** as an experimental unified missing value indicator across all types.

### 7.1 The Peculiar Nature of `np.nan`
In standard computer science, `NaN` is not equal to anything — **not even itself**!
This means you can NEVER check for missing data using `if x == np.nan:`. You MUST use `pd.isna()`!

In [24]:
print("Is np.nan == np.nan?", np.nan == np.nan) # Returns False!
print("Is pd.isna(np.nan)? ", pd.isna(np.nan))   # Returns True!

Is np.nan == np.nan? False
Is pd.isna(np.nan)?  True


Let's build a realistic customer dataset containing various missing fields:

In [25]:
customer_data = {
    'Customer_ID': [1001, 1002, 1003, 1004, 1005, 1006],
    'Name': ['Acme Corp', 'Globex', 'Soylent Corp', 'Initech', 'Umbrella Corp', 'Hooli'],
    'Orders_Count': [14, np.nan, 8, 25, np.nan, 31],
    'Annual_Spend': [14000.0, 45000.0, np.nan, 32000.0, np.nan, 68000.0],
    'Satisfaction_Score': [4.5, 3.8, 4.2, np.nan, np.nan, 4.9],
    'Region': ['North', 'East', np.nan, 'West', 'East', 'North']
}

df_cust = pd.DataFrame(customer_data)
print("Raw Customer Dataset with Missing Values:")
display(df_cust)

Raw Customer Dataset with Missing Values:


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
0,1001,Acme Corp,14.0,14000.0,4.5,North
1,1002,Globex,NaN,45000.0,3.8,East
2,1003,Soylent Corp,8.0,NaN,4.2,NaN
3,1004,Initech,25.0,32000.0,NaN,West
4,1005,Umbrella Corp,NaN,NaN,NaN,East
5,1006,Hooli,31.0,68000.0,4.9,North


### 7.2 Detecting & Quantifying Missing Data
Use `.isna()` to flag missing values, and `.sum()` to count them per column:

In [26]:
# 1. Total missing values per column
print("Missing values per column:")
print(df_cust.isna().sum())

# 2. Percentage of missing data per column
print("\nPercentage of missing data per column:")
print((df_cust.isna().mean() * 100).round(1).astype(str) + '%')

# 3. Overall total missing values across entire table
print(f"\nTotal missing cells in table: {df_cust.isna().sum().sum()}")

Missing values per column:
Customer_ID           0
Name                  0
Orders_Count          2
Annual_Spend          2
Satisfaction_Score    2
Region                1
dtype: int64

Percentage of missing data per column:
Customer_ID            0.0%
Name                   0.0%
Orders_Count          33.3%
Annual_Spend          33.3%
Satisfaction_Score    33.3%
Region                16.7%
dtype: str

Total missing cells in table: 7


### 7.3 Dropping Missing Values with `.dropna()`
Dropping rows is appropriate when only a tiny fraction of data is missing, or when critical identifier columns are empty.
Key parameters:
- `axis=0` (default): Drop rows containing missing values.
- `how='any'` (default): Drop row if AT LEAST ONE column has NaN.
- `how='all'`: Drop row only if ALL columns are NaN.
- `subset=['ColA', 'ColB']`: Only consider NaNs in these specified columns.

In [27]:
# Drop rows where ANY column is missing
print("--- Drop ANY Missing Value ---")
display(df_cust.dropna())

# Drop rows where 'Annual_Spend' specifically is missing
print("\n--- Drop Only If Annual_Spend is Missing ---")
display(df_cust.dropna(subset=['Annual_Spend']))

--- Drop ANY Missing Value ---


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
0,1001,Acme Corp,14.0,14000.0,4.5,North
5,1006,Hooli,31.0,68000.0,4.9,North



--- Drop Only If Annual_Spend is Missing ---


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
0,1001,Acme Corp,14.0,14000.0,4.5,North
1,1002,Globex,NaN,45000.0,3.8,East
3,1004,Initech,25.0,32000.0,NaN,West
5,1006,Hooli,31.0,68000.0,4.9,North


### 7.4 Imputing Missing Values with `.fillna()`
Throwing away data can bias analysis or destroy valuable sample size. **Imputation** replaces missing values with informed approximations:
- **Numerical columns:** Replace with column `mean()` (symmetric data) or `median()` (skewed data with outliers).
- **Categorical columns:** Replace with `'Unknown'` or the `mode()` (most frequent category).
- **Time-series data:** Forward fill with `.ffill()` (carry previous known value forward) or backward fill with `.bfill()`.

In [28]:
df_clean = df_cust.copy()

# 1. Impute categorical column with constant
df_clean['Region'] = df_clean['Region'].fillna('Unknown')

# 2. Impute count with 0
df_clean['Orders_Count'] = df_clean['Orders_Count'].fillna(0)

# 3. Impute Annual_Spend with column median
spend_median = df_clean['Annual_Spend'].median()
df_clean['Annual_Spend'] = df_clean['Annual_Spend'].fillna(spend_median)

# 4. Impute Satisfaction_Score with column mean
satisfaction_mean = df_clean['Satisfaction_Score'].mean()
df_clean['Satisfaction_Score'] = df_clean['Satisfaction_Score'].fillna(satisfaction_mean).round(2)

print(f"Imputed spend with median: ${spend_median:,.2f}")
print(f"Imputed satisfaction with mean: {satisfaction_mean:.2f}")
print("\nFully Imputed Clean DataFrame:")
display(df_clean)

Imputed spend with median: $38,500.00
Imputed satisfaction with mean: 4.35

Fully Imputed Clean DataFrame:


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
0,1001,Acme Corp,14.0,14000.0,4.50,North
1,1002,Globex,0.0,45000.0,3.80,East
2,1003,Soylent Corp,8.0,38500.0,4.20,Unknown
3,1004,Initech,25.0,32000.0,4.35,West
4,1005,Umbrella Corp,0.0,38500.0,4.35,East
5,1006,Hooli,31.0,68000.0,4.90,North


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch8"></a>
---
## 8. Sorting, Ranking & Index Manipulation

### 8.1 Sorting by Values
Use `.sort_values(by=..., ascending=...)`:
- Pass a single column name to sort by one attribute.
- Pass a list of column names and a corresponding list of booleans to `ascending` to execute multi-level sorting (e.g. sort Department A–Z, then Salary High-to-Low within each department).

In [29]:
# Single column sort (Highest spend first)
sorted_spend = df_clean.sort_values(by='Annual_Spend', ascending=False)
print("Customers sorted by Annual Spend (Descending):")
display(sorted_spend.head(3))

# Multi-column sort: Region (Ascending), then Annual_Spend (Descending)
multi_sort = df_clean.sort_values(
    by=['Region', 'Annual_Spend'],
    ascending=[True, False]
)
print("\nSorted by Region (A-Z), then Spend (High to Low):")
display(multi_sort)

Customers sorted by Annual Spend (Descending):


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
5,1006,Hooli,31.0,68000.0,4.9,North
1,1002,Globex,0.0,45000.0,3.8,East
2,1003,Soylent Corp,8.0,38500.0,4.2,Unknown



Sorted by Region (A-Z), then Spend (High to Low):


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
1,1002,Globex,0.0,45000.0,3.80,East
4,1005,Umbrella Corp,0.0,38500.0,4.35,East
5,1006,Hooli,31.0,68000.0,4.90,North
0,1001,Acme Corp,14.0,14000.0,4.50,North
2,1003,Soylent Corp,8.0,38500.0,4.20,Unknown
3,1004,Initech,25.0,32000.0,4.35,West


### 8.2 Ranking Values with `.rank()`
Sorting reorders rows, whereas `.rank()` assigns numerical standings ($1, 2, 3, \dots$) without changing row order:
- `method='average'`: Default; assigns the average rank to ties.
- `method='min'`: Competition ranking (e.g. Olympic medals: two silver medalists tied for 2nd means next place is 4th).
- `method='dense'`: Like `min`, but does not leave gaps after ties (next place is 3rd).

In [30]:
df_ranked = df_clean.copy()
df_ranked['Spend_Rank'] = df_ranked['Annual_Spend'].rank(ascending=False, method='min').astype(int)
display(df_ranked[['Name', 'Annual_Spend', 'Spend_Rank']].sort_values(by='Spend_Rank'))

,Name,Annual_Spend,Spend_Rank
5,Hooli,68000.0,1
1,Globex,45000.0,2
2,Soylent Corp,38500.0,3
4,Umbrella Corp,38500.0,3
3,Initech,32000.0,5
0,Acme Corp,14000.0,6


### 8.3 Resetting & Setting the Index
After sorting or filtering, row index labels become scrambled (e.g. 5, 1, 3, 0).
- **`.reset_index(drop=True)`**: Discards old scrambled labels and generates a clean sequence starting at 0.
- **`.set_index('Column_Name')`**: Promotes an existing column into the row index.

In [31]:
# Resetting index after sorting
clean_reindexed = sorted_spend.reset_index(drop=True)
print("Index reset cleanly from 0 to N-1:")
display(clean_reindexed.head(3))

# Setting Customer_ID as the index
df_id_index = clean_reindexed.set_index('Customer_ID')
print("\nDataFrame indexed by Customer_ID:")
display(df_id_index.head(2))

Index reset cleanly from 0 to N-1:


,Customer_ID,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
0,1006,Hooli,31.0,68000.0,4.9,North
1,1002,Globex,0.0,45000.0,3.8,East
2,1003,Soylent Corp,8.0,38500.0,4.2,Unknown



DataFrame indexed by Customer_ID:


,Name,Orders_Count,Annual_Spend,Satisfaction_Score,Region
Customer_ID,,,,,
1006,Hooli,31.0,68000.0,4.9,North
1002,Globex,0.0,45000.0,3.8,East


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch9"></a>
---
# Part IV: Summaries & Group Operations

## 9. Summary Statistics & Frequencies

Pandas provides rich aggregate methods to describe numerical and categorical distributions instantly.

### 9.1 Summary Statistics
By default, aggregate operations automatically ignore `NaN` values:

| Method | What It Computes |
| :--- | :--- |
| `df['col'].mean()` | Arithmetic average |
| `df['col'].median()` | 50th percentile (robust to outliers) |
| `df['col'].std()` | Sample standard deviation |
| `df['col'].var()` | Sample variance |
| `df['col'].min()`, `max()` | Extreme boundaries |
| `df['col'].quantile(0.75)` | Arbitrary quantile (e.g., 75th percentile) |
| `df['col'].sum()` | Total accumulation |

In [32]:
print(f"Mean Spend:         ${df_clean['Annual_Spend'].mean():,.2f}")
print(f"Median Spend:       ${df_clean['Annual_Spend'].median():,.2f}")
print(f"Standard Deviation: ${df_clean['Annual_Spend'].std():,.2f}")
print(f"25th Percentile:    ${df_clean['Annual_Spend'].quantile(0.25):,.2f}")
print(f"75th Percentile:    ${df_clean['Annual_Spend'].quantile(0.75):,.2f}")
print(f"Interquartile Range: ${df_clean['Annual_Spend'].quantile(0.75) - df_clean['Annual_Spend'].quantile(0.25):,.2f}")

Mean Spend:         $39,333.33
Median Spend:       $38,500.00
Standard Deviation: $17,611.55
25th Percentile:    $33,625.00
75th Percentile:    $43,375.00
Interquartile Range: $9,750.00


### 9.2 Categorical Frequencies with `.value_counts()`
`.value_counts()` is one of the most frequently used methods in Pandas. It computes the frequency of each unique value in a Series:
- Pass `normalize=True` to compute relative proportions (percentages).
- Pass `dropna=False` to include counts of missing values.

In [33]:
# Frequency counts of Regions
print("Customer Count per Region:")
print(df_clean['Region'].value_counts())

# Relative proportion percentage
print("\nPercentage Distribution per Region:")
print((df_clean['Region'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

# Unique count and unique values
print(f"\nNumber of unique regions: {df_clean['Region'].nunique()}")
print(f"Unique region labels:     {df_clean['Region'].unique()}")

Customer Count per Region:
Region
North      2
East       2
Unknown    1
West       1
Name: count, dtype: int64

Percentage Distribution per Region:
Region
North      33.3%
East       33.3%
Unknown    16.7%
West       16.7%
Name: proportion, dtype: str

Number of unique regions: 4
Unique region labels:     <StringArray>
['North', 'East', 'Unknown', 'West']
Length: 4, dtype: str


### 9.3 Correlation Matrix with `.corr()`
Examine linear relationships between numerical features using Pearson correlation:

In [34]:
# Numerical features correlation matrix
numeric_cols = ['Orders_Count', 'Annual_Spend', 'Satisfaction_Score']
corr_matrix = df_clean[numeric_cols].corr()
print("Correlation Matrix:")
display(corr_matrix)

Correlation Matrix:


,Orders_Count,Annual_Spend,Satisfaction_Score
Orders_Count,1.000000,0.302841,0.771873
Annual_Spend,0.302841,1.000000,0.282680
Satisfaction_Score,0.771873,0.282680,1.000000


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch10"></a>
---
## 10. The GroupBy Engine (Split-Apply-Combine)

The `groupby()` engine is the centerpiece of tabular data analysis, analogous to SQL's `GROUP BY` clause.

### 10.1 The Split-Apply-Combine Mental Model
Hadley Wickham coined the term **Split-Apply-Combine** to describe group operations:
1. **SPLIT:** Divide the DataFrame into separate subsets based on keys (e.g. Department).
2. **APPLY:** Compute a summary statistic (e.g. `mean`, `sum`, `count`) independently on each subset.
3. **COMBINE:** Stitch the individual group results back into a clean output DataFrame.

```
       ORIGINAL DATA                    1. SPLIT                     2. APPLY               3. COMBINE
+------------+-----------+      +------------+-----------+
| Department | Salary    |      | Department | Salary    |
+------------+-----------+      +------------+-----------+
| IT         |  88,000   | ---> | IT         |  88,000   | --->  mean(IT): 91k  ---> +------------+----------+
| HR         |  62,000   |      | IT         |  94,000   |                           | Department | MeanSal  |
| IT         |  94,000   |      +------------+-----------+                           +------------+----------+
| Marketing  |  71,000   |      | HR         |  62,000   | --->  mean(HR): 62k       | Finance    | 105,000  |
| Finance    | 105,000   |      +------------+-----------+                           | HR         |  62,000  |
+------------+-----------+      | Marketing  |  71,000   | --->  mean(Mktg): 71k     | IT         |  91,000  |
                                +------------+-----------+                           | Marketing  |  71,000  |
                                | Finance    | 105,000   | --->  mean(Fin): 105k     +------------+----------+
                                +------------+-----------+
```

In [35]:
# Simple single-column aggregation
avg_dept_salary = df_staff.groupby('Department')['Salary'].mean()
print("Average Salary by Department:")
print(avg_dept_salary)

Average Salary by Department:
Department
Finance      105000.0
HR            62000.0
IT            91000.0
Marketing     71000.0
Name: Salary, dtype: float64


### 10.2 Multiple Aggregations with `.agg()`: Named Aggregations
Often you want to calculate multiple metrics simultaneously (e.g., headcount, average salary, minimum, and maximum salary).
Modern Pandas offers **Named Aggregation** syntax: `agg(New_Col_Name=('Source_Col', 'agg_func'))`. This produces clean, un-nested column names!

In [36]:
# Clean named aggregations
dept_summary = df_staff.groupby('Department').agg(
    Headcount=('Emp_ID', 'count'),
    Avg_Salary=('Salary', 'mean'),
    Min_Salary=('Salary', 'min'),
    Max_Salary=('Salary', 'max'),
    Avg_Experience=('Years_Experience', 'mean')
).reset_index()

print("Department Summary Table:")
display(dept_summary)

Department Summary Table:


,Department,Headcount,Avg_Salary,Min_Salary,Max_Salary,Avg_Experience
0,Finance,1,105000.0,105000,105000,12.0
1,HR,1,62000.0,62000,62000,3.0
2,IT,2,91000.0,88000,94000,8.0
3,Marketing,1,71000.0,71000,71000,4.0


### 10.3 GroupBy `.transform()`: Broadcast Group Metrics to Original Rows
Unlike `.agg()` which collapses rows into group summaries, **`.transform()`** computes a metric per group but returns a Series with the **exact same length and index as the original DataFrame**!
This makes it effortless to compare an individual record against its group average (e.g. "How much more does Alice earn compared to her department's average?").

In [37]:
df_analysis = df_staff.copy()

# Compute department average salary and attach it directly to each employee's row
df_analysis['Dept_Avg_Salary'] = df_analysis.groupby('Department')['Salary'].transform('mean')

# Calculate deviation from department average
df_analysis['Diff_From_Dept_Avg'] = df_analysis['Salary'] - df_analysis['Dept_Avg_Salary']

display(df_analysis[['Name', 'Department', 'Salary', 'Dept_Avg_Salary', 'Diff_From_Dept_Avg']])

,Name,Department,Salary,Dept_Avg_Salary,Diff_From_Dept_Avg
0,Alice Smith,HR,62000,62000.0,0.0
1,Bob Jones,IT,88000,91000.0,-3000.0
2,Charlie Brown,Marketing,71000,71000.0,0.0
3,Diana Prince,IT,94000,91000.0,3000.0
4,Evan Wright,Finance,105000,105000.0,0.0


### 10.4 GroupBy `.filter()`: Discard Entire Groups Based on Group Criteria
You can filter out groups that don't meet an aggregate threshold (e.g. "Only keep departments that have 2 or more employees"):

In [38]:
# Keep only departments with at least 2 employees
large_depts = df_staff.groupby('Department').filter(lambda group: len(group) >= 2)
print("Employees belonging to departments with 2+ staff:")
display(large_depts[['Name', 'Department', 'Salary']])

Employees belonging to departments with 2+ staff:


,Name,Department,Salary
1,Bob Jones,IT,88000
3,Diana Prince,IT,94000


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch11"></a>
---
# Part V: Real-World Techniques

## 11. Text & String Operations (`.str`)

Real-world text data is notoriously messy: inconsistent casing, extraneous whitespace, and mixed delimiters.
Pandas provides a dedicated **`.str` accessor** to execute vectorized string operations across entire columns without Python loops.

Let's inspect messy customer feedback data:

In [39]:
feedback_data = {
    'Feedback_ID': [1, 2, 3, 4],
    'Raw_Customer': ['   alice smith   ', 'BOB JONES', 'charlie brown  ', 'diana prince'],
    'Email': ['ALICE@GMAIL.COM', 'bob.j@company.org', 'charlie_b@yahoo.com', 'diana@COMPANY.ORG'],
    'Review': [
        'Great product! Fast delivery and great support.',
        'Package was damaged upon arrival. Very poor.',
        'Excellent quality, will definitely order again.',
        'Decent product, but delivery was slow.'
    ]
}

df_fb = pd.DataFrame(feedback_data)
display(df_fb)

,Feedback_ID,Raw_Customer,Email,Review
0,1,alice smith,ALICE@GMAIL.COM,Great product! Fast delivery and great support.
1,2,BOB JONES,bob.j@company.org,Package was damaged upon arrival. Very poor.
2,3,charlie brown,charlie_b@yahoo.com,"Excellent quality, will definitely order again."
3,4,diana prince,diana@COMPANY.ORG,"Decent product, but delivery was slow."


### 11.1 Cleaning Text: Strip, Lower, Title Case
You can chain string methods together cleanly:

In [40]:
# 1. Clean whitespace and convert names to Title Case
df_fb['Clean_Name'] = df_fb['Raw_Customer'].str.strip().str.title()

# 2. Standardize emails to lowercase
df_fb['Clean_Email'] = df_fb['Email'].str.strip().str.lower()

display(df_fb[['Clean_Name', 'Clean_Email']])

,Clean_Name,Clean_Email
0,Alice Smith,alice@gmail.com
1,Bob Jones,bob.j@company.org
2,Charlie Brown,charlie_b@yahoo.com
3,Diana Prince,diana@company.org


### 11.2 Pattern Matching with `.str.contains()`
Search for substring occurrences or regular expressions. Pass `case=False` for case-insensitive matching:

In [41]:
# Flag positive reviews containing 'great' or 'excellent'
pattern = r'great|excellent'
df_fb['Is_Positive'] = df_fb['Review'].str.contains(pattern, case=False, regex=True)

display(df_fb[['Clean_Name', 'Review', 'Is_Positive']])

,Clean_Name,Review,Is_Positive
0,Alice Smith,Great product! Fast delivery and great support.,True
1,Bob Jones,Package was damaged upon arrival. Very poor.,False
2,Charlie Brown,"Excellent quality, will definitely order again.",True
3,Diana Prince,"Decent product, but delivery was slow.",False


### 11.3 Splitting & Regex Extraction with `expand=True`
- `.str.split('@', expand=True)` splits strings into separate DataFrame columns.
- `.str.extract(pattern)` extracts regex capture groups into named columns.

In [42]:
# Extract username and domain from email
split_email = df_fb['Clean_Email'].str.split('@', expand=True)
split_email.columns = ['Email_Username', 'Email_Domain']

# Combine with main DataFrame
df_fb_expanded = pd.concat([df_fb[['Clean_Name', 'Clean_Email']], split_email], axis=1)
display(df_fb_expanded)

,Clean_Name,Clean_Email,Email_Username,Email_Domain
0,Alice Smith,alice@gmail.com,alice,gmail.com
1,Bob Jones,bob.j@company.org,bob.j,company.org
2,Charlie Brown,charlie_b@yahoo.com,charlie_b,yahoo.com
3,Diana Prince,diana@company.org,diana,company.org


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch12"></a>
---
## 12. Working with Dates & Times (`.dt`)

Date and time information frequently arrives as plain string text (`'2026-03-15'`). Converting text into proper datetime objects is crucial for chronological sorting, time math, and calendar feature engineering.

### 12.1 Parsing Datetimes with `pd.to_datetime()`
`pd.to_datetime()` automatically handles most ISO 8601 string formats. Specifying the exact `format` argument significantly speeds up parsing on large files:

In [43]:
order_records = {
    'Order_ID': [5001, 5002, 5003, 5004, 5005],
    'Order_Timestamp': [
        '2026-01-15 09:30:00',
        '2026-01-18 14:15:00',
        '2026-02-05 18:45:00',
        '2026-03-12 11:20:00',
        '2026-03-29 16:50:00'
    ],
    'Delivery_Timestamp': [
        '2026-01-18 12:00:00',
        '2026-01-22 17:30:00',
        '2026-02-07 10:15:00',
        '2026-03-16 15:00:00',
        '2026-04-03 09:30:00'
    ],
    'Total_Amount': [145.0, 230.5, 89.0, 412.0, 310.0]
}

df_orders = pd.DataFrame(order_records)
print("Before conversion:")
print(df_orders.dtypes)

# Convert strings to datetime64[ns]
df_orders['Order_Timestamp'] = pd.to_datetime(df_orders['Order_Timestamp'])
df_orders['Delivery_Timestamp'] = pd.to_datetime(df_orders['Delivery_Timestamp'])

print("\nAfter conversion:")
print(df_orders.dtypes)

Before conversion:
Order_ID                int64
Order_Timestamp           str
Delivery_Timestamp        str
Total_Amount          float64
dtype: object

After conversion:
Order_ID                       int64
Order_Timestamp       datetime64[us]
Delivery_Timestamp    datetime64[us]
Total_Amount                 float64
dtype: object


### 12.2 Extracting Calendar Properties with the `.dt` Accessor
Similar to `.str`, the **`.dt` accessor** exposes calendar properties:
- `.dt.year`, `.dt.month`, `.dt.day`
- `.dt.day_name()`, `.dt.month_name()`
- `.dt.quarter`, `.dt.dayofweek` (0 = Monday, 6 = Sunday)
- `.dt.is_month_end`, `.dt.is_weekend`

In [44]:
# Calendar feature engineering
df_orders['Order_Year'] = df_orders['Order_Timestamp'].dt.year
df_orders['Order_Month'] = df_orders['Order_Timestamp'].dt.month_name()
df_orders['Order_Day_Name'] = df_orders['Order_Timestamp'].dt.day_name()
df_orders['Is_Weekend'] = df_orders['Order_Timestamp'].dt.dayofweek >= 5

display(df_orders[['Order_ID', 'Order_Timestamp', 'Order_Month', 'Order_Day_Name', 'Is_Weekend']])

,Order_ID,Order_Timestamp,Order_Month,Order_Day_Name,Is_Weekend
0,5001,2026-01-15 09:30:00,January,Thursday,False
1,5002,2026-01-18 14:15:00,January,Sunday,True
2,5003,2026-02-05 18:45:00,February,Thursday,False
3,5004,2026-03-12 11:20:00,March,Thursday,False
4,5005,2026-03-29 16:50:00,March,Sunday,True


### 12.3 Date Arithmetic & Timedeltas
Subtracting two datetime columns produces a **`Timedelta`** Series, allowing exact duration measurements:

In [45]:
# Calculate delivery duration
df_orders['Delivery_Duration'] = df_orders['Delivery_Timestamp'] - df_orders['Order_Timestamp']

# Extract fractional days or total hours
df_orders['Duration_Days'] = df_orders['Delivery_Duration'].dt.total_seconds() / 86400

print("Order Delivery Durations:")
display(df_orders[['Order_ID', 'Delivery_Duration', 'Duration_Days']])
print(f"\nAverage delivery duration: {df_orders['Duration_Days'].mean():.2f} days")

Order Delivery Durations:


,Order_ID,Delivery_Duration,Duration_Days
0,5001,3 days 02:30:00,3.104167
1,5002,4 days 03:15:00,4.135417
2,5003,1 days 15:30:00,1.645833
3,5004,4 days 03:40:00,4.152778
4,5005,4 days 16:40:00,4.694444



Average delivery duration: 3.55 days


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch13"></a>
---
## 13. Combining Datasets: Concat & Merge

In production, data is normalized across multiple tables. Pandas provides two core mechanisms for combining data:
1. **`pd.concat()`**: Stacking DataFrames on top of each other (rows) or side-by-side (columns).
2. **`pd.merge()`**: Relational SQL-style joins matching on common key columns.

### 13.1 Concatenation: Stacking DataFrames
Use `pd.concat([df1, df2], axis=0)` to combine monthly or quarterly files that share the same schema:

In [46]:
# Monthly sales tables
jan_sales = pd.DataFrame({'Product': ['Laptop', 'Phone'], 'Revenue': [12000, 8000]})
feb_sales = pd.DataFrame({'Product': ['Laptop', 'Tablet'], 'Revenue': [14000, 6000]})

# Stack rows vertically (axis=0)
combined_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)
print("Combined Sales (Row-stacked):")
display(combined_sales)

Combined Sales (Row-stacked):


,Product,Revenue
0,Laptop,12000
1,Phone,8000
2,Laptop,14000
3,Tablet,6000


### 13.2 Merging: Relational SQL Joins
`pd.merge(left, right, on='Key_Col', how='inner'|'left'|'right'|'outer')`

```
                      VISUAL GUIDE TO PANDAS MERGES
+-------------------+--------------------+-------------------+--------------------+
|    INNER JOIN     |     LEFT JOIN      |    RIGHT JOIN     |     OUTER JOIN     |
|   how='inner'     |    how='left'      |    how='right'    |    how='outer'     |
+-------------------+--------------------+-------------------+--------------------+
| Only keys present | ALL rows from left | ALL rows from right| ALL rows from both|
| in BOTH tables.   | table. Unmatched   | table. Unmatched  | tables. Missing    |
|                   | right keys get NaN | left keys get NaN | values get NaN     |
+-------------------+--------------------+-------------------+--------------------+
```

In [47]:
# Table 1: Customers
customers_tbl = pd.DataFrame({
    'Cust_ID': [1, 2, 3, 4],
    'Company': ['Acme Corp', 'Globex', 'Soylent', 'Initech']
})

# Table 2: Orders (Note: Cust_ID 5 is an unregistered customer; Cust_ID 4 has placed no orders)
orders_tbl = pd.DataFrame({
    'Order_ID': [901, 902, 903, 904],
    'Cust_ID': [1, 2, 1, 5],
    'Amount': [450.0, 1200.0, 310.0, 890.0]
})

print("Customers Table:")
display(customers_tbl)
print("\nOrders Table:")
display(orders_tbl)

Customers Table:


,Cust_ID,Company
0,1,Acme Corp
1,2,Globex
2,3,Soylent
3,4,Initech



Orders Table:


,Order_ID,Cust_ID,Amount
0,901,1,450.0
1,902,2,1200.0
2,903,1,310.0
3,904,5,890.0


In [48]:
# 1. Inner Join (Matches only)
inner_join = pd.merge(customers_tbl, orders_tbl, on='Cust_ID', how='inner')
print("--- Inner Join (Only Matching Customers & Orders) ---")
display(inner_join)

# 2. Left Join (Keep all customers)
left_join = pd.merge(customers_tbl, orders_tbl, on='Cust_ID', how='left')
print("\n--- Left Join (Preserves All Customers) ---")
display(left_join)

# 3. Outer Join (Full union)
outer_join = pd.merge(customers_tbl, orders_tbl, on='Cust_ID', how='outer')
print("\n--- Outer Join (Full Union of Both Tables) ---")
display(outer_join)

--- Inner Join (Only Matching Customers & Orders) ---


,Cust_ID,Company,Order_ID,Amount
0,1,Acme Corp,901,450.0
1,1,Acme Corp,903,310.0
2,2,Globex,902,1200.0



--- Left Join (Preserves All Customers) ---


,Cust_ID,Company,Order_ID,Amount
0,1,Acme Corp,901.0,450.0
1,1,Acme Corp,903.0,310.0
2,2,Globex,902.0,1200.0
3,3,Soylent,NaN,NaN
4,4,Initech,NaN,NaN



--- Outer Join (Full Union of Both Tables) ---


,Cust_ID,Company,Order_ID,Amount
0,1,Acme Corp,901.0,450.0
1,1,Acme Corp,903.0,310.0
2,2,Globex,902.0,1200.0
3,3,Soylent,NaN,NaN
4,4,Initech,NaN,NaN
5,5,NaN,904.0,890.0


### 13.3 Detecting Non-Matches with the Anti-Join Pattern
To find customers who have NEVER placed an order, perform a `left` merge with `indicator=True`:

In [49]:
anti_join_check = pd.merge(customers_tbl, orders_tbl, on='Cust_ID', how='left', indicator=True)
unmatched_customers = anti_join_check[anti_join_check['_merge'] == 'left_only']

print("Customers who have NEVER placed an order (Anti-Join):")
display(unmatched_customers[['Cust_ID', 'Company']])

Customers who have NEVER placed an order (Anti-Join):


,Cust_ID,Company
3,3,Soylent
4,4,Initech


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch14"></a>
---
## 14. Reshaping: Pivot Tables & Melt

Data is often formatted for human readability (wide tables with months across columns) rather than machine learning (tidy, tall tables where each row is an observation).

### 14.1 Pivot Tables: 2-Dimensional Summaries
`df.pivot_table()` is identical to Excel pivot tables:
- `index`: Column to use as row headers.
- `columns`: Column to use as column headers.
- `values`: Column to aggregate.
- `aggfunc`: Function to aggregate with (default is `'mean'`, can use `'sum'`, `'count'`).
- `margins=True`: Appends subtotal/total rows and columns.

In [50]:
store_sales = pd.DataFrame({
    'Store': ['Downtown', 'Suburbs', 'Downtown', 'Suburbs', 'Downtown', 'Suburbs'],
    'Department': ['Electronics', 'Electronics', 'Clothing', 'Clothing', 'Electronics', 'Clothing'],
    'Quarter': ['Q1', 'Q1', 'Q1', 'Q1', 'Q2', 'Q2'],
    'Sales': [15000, 12000, 8000, 9500, 18000, 11000]
})

# Pivot Table: Store vs Department Total Sales
pivot_table_res = store_sales.pivot_table(
    index='Store',
    columns='Department',
    values='Sales',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

print("Sales Pivot Table (Store by Department):")
display(pivot_table_res)

Sales Pivot Table (Store by Department):

Department,Clothing,Electronics,Total
Store,,,
Downtown,8000,33000,41000
Suburbs,20500,12000,32500
Total,28500,45000,73500


### 14.2 Unpivoting with `pd.melt()` (Wide to Long Format)
Machine learning libraries (Scikit-Learn, PyTorch) require **tidy data**, where each column represents a single variable.
`pd.melt()` unpivots a wide table into a tall, tidy format:

In [51]:
# Wide table (Months are column headers)
wide_revenue = pd.DataFrame({
    'Region': ['North', 'South'],
    'Jan_Rev': [10000, 12000],
    'Feb_Rev': [11500, 12500],
    'Mar_Rev': [13000, 14000]
})

print("Wide Format Table:")
display(wide_revenue)

# Melt into long tidy format
tidy_revenue = pd.melt(
    wide_revenue,
    id_vars=['Region'],
    value_vars=['Jan_Rev', 'Feb_Rev', 'Mar_Rev'],
    var_name='Month',
    value_name='Revenue'
)

print("\nTidy Long Format Table (via pd.melt):")
display(tidy_revenue)

Wide Format Table:


,Region,Jan_Rev,Feb_Rev,Mar_Rev
0,North,10000,11500,13000
1,South,12000,12500,14000



Tidy Long Format Table (via pd.melt):

,Region,Month,Revenue
0,North,Jan_Rev,10000
1,South,Jan_Rev,12000
2,North,Feb_Rev,11500
3,South,Feb_Rev,12500
4,North,Mar_Rev,13000
5,South,Mar_Rev,14000


> [⬆️ Back to Table of Contents](#table-of-contents)

<a id="ch15"></a>
---
## 15. Performance, Gotchas & Cheat Sheet

### 15.1 The Dreaded `SettingWithCopyWarning`
Every Pandas user encounters this warning:
`SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame.`

#### Why does it occur?
It happens during **chained assignment** like `df[df['Score'] > 80]['Passed'] = True`:
- Step 1: `df[df['Score'] > 80]` extracts a subset.
- Step 2: `['Passed'] = True` attempts to write to that subset.
- The Problem: Pandas cannot determine whether the subset is a **view** (sharing memory with the original) or a **copy** (an independent buffer). If it's a copy, your assignment is silently lost!

#### The Two Best Solutions:
1. **If you want to modify the original DataFrame:** Use `.loc[row_condition, 'col'] = value`.
2. **If you want a standalone new DataFrame:** Explicitly call `.copy()`.

In [52]:
sample_scores = pd.DataFrame({'Student': ['Amy', 'Ben', 'Chris'], 'Score': [85, 62, 91]})

# BEST PRACTICE 1: Use .loc to modify the original table directly
sample_scores.loc[sample_scores['Score'] >= 70, 'Status'] = 'Pass'
sample_scores.loc[sample_scores['Score'] < 70, 'Status'] = 'Fail'
print("Safely modified original using .loc:")
display(sample_scores)

# BEST PRACTICE 2: Use .copy() to create an independent subset
passed_students = sample_scores[sample_scores['Status'] == 'Pass'].copy()
passed_students['Honors'] = passed_students['Score'] >= 90
print("\nSafely created independent subset using .copy():")
display(passed_students)

Safely modified original using .loc:


,Student,Score,Status
0,Amy,85,Pass
1,Ben,62,Fail
2,Chris,91,Pass



Safely created independent subset using .copy():


,Student,Score,Status,Honors
0,Amy,85,Pass,False
2,Chris,91,Pass,True


### 15.2 Execution Speed Benchmark: Vectorization vs `.apply()` vs `for` Loop
Never iterate through rows using Python `for` loops or `df.iterrows()` when a vectorized method exists.
Let's benchmark on **500,000 records**:

In [53]:
import time

# Create benchmark dataset of 500,000 rows
N = 500_000
bench_df = pd.DataFrame({
    'Base_Price': np.random.uniform(20.0, 200.0, size=N),
    'Tax_Rate': np.random.uniform(0.05, 0.20, size=N)
})

# 1. Vectorized Arithmetic (Fastest, C/SIMD speed)
t0 = time.perf_counter()
res_vec = bench_df['Base_Price'] * (1 + bench_df['Tax_Rate'])
t1 = time.perf_counter()
t_vectorized = t1 - t0

# 2. .apply(axis=1) with Python Lambda (Interpreted in Python)
# Benchmarked on 50,000 rows and extrapolated to 500k to save execution time
t0 = time.perf_counter()
_ = bench_df.iloc[:50_000].apply(lambda row: row['Base_Price'] * (1 + row['Tax_Rate']), axis=1)
t1 = time.perf_counter()
t_apply = (t1 - t0) * 10

print(f"🚀 Vectorized computation (500k): {t_vectorized:.4f} seconds")
print(f"🐢 .apply(axis=1) (extrapolated): {t_apply:.4f} seconds")
print(f"⚡ Vectorization is {t_apply / t_vectorized:.1f}x faster!")

🚀 Vectorized computation (500k): 0.0024 seconds


🐢 .apply(axis=1) (extrapolated): 1.6081 seconds
⚡ Vectorization is 670.1x faster!


### 15.3 Memory Optimization: The Power of Categorical Dtypes
By default, text columns are stored as `object` (arrays of pointers to Python string heap objects).
If a text column contains repeated values (e.g. Gender, Country, Department, Status), converting it to `'category'` stores the distinct strings once in a lookup table and replaces the column with tiny integers (e.g. `int8`).

In [54]:
# 200,000 rows with repeated city names
n_records = 200_000
cities = np.random.choice(['New York', 'London', 'Tokyo', 'Paris', 'Sydney'], size=n_records)
df_mem = pd.DataFrame({'City': cities})

# Measure memory
mem_object = df_mem['City'].memory_usage(deep=True)
df_mem['City'] = df_mem['City'].astype('category')
mem_category = df_mem['City'].memory_usage(deep=True)

print(f"📦 Object dtype memory:    {mem_object:>10,} bytes")
print(f"⚡ Category dtype memory:  {mem_category:>10,} bytes")
print(f"🎉 Memory reduced by {mem_object / mem_category:.1f}x!")

📦 Object dtype memory:    10,999,276 bytes
⚡ Category dtype memory:     200,407 bytes
🎉 Memory reduced by 54.9x!


---
<a id="summary"></a>
## 🏆 Comprehensive High-Frequency Cheat Sheet

| Task | Syntax / Method | Example |
| :--- | :--- | :--- |
| **Inspect Data** | `head()`, `tail()`, `info()`, `describe()` | `df.info()`, `df.describe()` |
| **Select Column(s)** | `df['col']` (Series) / `df[['c1', 'c2']]` (DataFrame) | `df[['Age', 'Salary']]` |
| **Select by Position** | `df.iloc[row_idx, col_idx]` | `df.iloc[0:5, 1:4]` |
| **Select by Label** | `df.loc[row_label, col_label]` | `df.loc['Alice':'Diana', 'Salary']` |
| **Filter by Condition** | `df[(cond1) & (cond2)]` | `df[(df['Age'] > 25) & (df['Remote'])]` |
| **Membership Filter** | `df['col'].isin([val1, val2])` | `df[df['Dept'].isin(['HR', 'IT'])]` |
| **Range Filter** | `df['col'].between(low, high)` | `df[df['Salary'].between(60k, 90k)]` |
| **Add / Modify Column** | `df['new'] = expression` | `df['Bonus'] = df['Salary'] * 0.1` |
| **Rename Column** | `df.rename(columns={'old': 'new'})` | `df.rename(columns={'id': 'user_id'})` |
| **Drop Column / Row** | `df.drop(columns=[...])` / `df.drop(index=[...])` | `df.drop(columns=['unneeded_col'])` |
| **Check Missing Data** | `df.isna().sum()` | `df.isna().sum()` |
| **Drop Missing Data** | `df.dropna(subset=[...])` | `df.dropna(subset=['critical_col'])` |
| **Fill Missing Data** | `df['col'].fillna(val)` | `df['Salary'].fillna(df['Salary'].median())` |
| **Sort Values** | `df.sort_values(by=..., ascending=...)` | `df.sort_values(by='Salary', ascending=False)` |
| **Reset Index** | `df.reset_index(drop=True)` | `df.reset_index(drop=True)` |
| **Value Counts** | `df['col'].value_counts(normalize=True)` | `df['Department'].value_counts()` |
| **GroupBy & Agg** | `df.groupby('key').agg(...)` | `df.groupby('dept').agg(avg_sal=('sal', 'mean'))` |
| **Transform Group** | `df.groupby('key')['col'].transform('mean')` | `df['dept_avg'] = df.groupby('dept')['sal'].transform('mean')` |
| **Clean Strings** | `df['text'].str.strip().str.title()` | `df['name'].str.strip().str.title()` |
| **Date Conversion** | `pd.to_datetime(df['date'])` | `df['date'] = pd.to_datetime(df['date'])` |
| **Date Properties** | `df['date'].dt.year`, `.dt.day_name()` | `df['weekday'] = df['date'].dt.day_name()` |
| **Stack Tables** | `pd.concat([df1, df2], ignore_index=True)` | `pd.concat([q1, q2], axis=0)` |
| **Join Tables** | `pd.merge(df1, df2, on='key', how='left')` | `pd.merge(users, orders, on='id')` |
| **Pivot Table** | `df.pivot_table(index=..., columns=..., values=...)`| `df.pivot_table(index='dept', columns='year', values='sal')` |
| **Melt Wide to Long** | `pd.melt(df, id_vars=[...], value_vars=[...])` | `pd.melt(df, id_vars=['id'], var_name='month')` |

---
### 🎯 Practice What You Have Learned!
Put your skills into practice with the companion exercise notebooks:
- 🏋️ [02_pandas_exercise_1.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/pandas/02_pandas_exercise_1.ipynb): 20 progressive practice problems.
- 💡 [03_pandas_exercise_1_solutions.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/pandas/03_pandas_exercise_1_solutions.ipynb): Verified solutions with pre-executed outputs.
- 🚀 [04_pandas_exercise_2.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/pandas/04_pandas_exercise_2.ipynb): 15 real-world data engineering and analytics scenario challenges.
- 💡 [05_pandas_exercise_2_solutions.ipynb](file:///Users/Inz_mac/Developer/ML/my-note/pandas/05_pandas_exercise_2_solutions.ipynb): Full scenario solutions with pre-executed outputs.